# Applying the Random Forest to 2025 Data

This notebook mirrors `modeling_2025data.ipynb`. It loads the model trained on 2019 data, scores every 2025 tract, and saves the predictions. It then compares the results with the logistic regression's 2025 predictions.

**Run `RF_risk_prediction_modeling.ipynb` first** so `rf_model.pkl` exists.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score, average_precision_score)

## Step 1: Load the saved model

The pickle file holds a dictionary, so we pull out the model, the feature list, and the threshold. `"rb"` means "read binary."

In [ ]:
with open("rf_model.pkl", "rb") as f:
    bundle = pickle.load(f)

model = bundle["model"]
FEATURES = bundle["features"]
THRESHOLD = bundle["threshold"]

print("Threshold:", THRESHOLD)
print("Best settings:", bundle["best_params"])

## Step 2: Load the 2025 data

The filtering is copied exactly from the logistic regression 2025 notebook.

In [ ]:
DATA_DIR = Path("../logistic_regression_modeling/final_data")

df = pd.read_csv(DATA_DIR / "cleaned_2025.csv")
df = df.iloc[:, :-4]
df = df[[i for i in df.columns if 'income' not in i]]
df = df[[i for i in df.columns if 'original' not in i]]
df = df[[i for i in df.columns if 'qa' not in i]]
df = df.drop(columns=['fd_label', 'label_status', 'label_reason'])

print(df.shape)
df.head()

## Step 3: Missing values

The logistic regression notebook filled these by hand using **2025 medians**. Here there's nothing to fill manually: the imputer inside the pipeline fills gaps with the **2019 training medians** it learned. That treats 2025 the way a real deployment would, using only what was known at training time.

**Missing target values:** 271 tracts have no `FD_2` label. The logistic regression notebook filled those with 0 ("not a food desert"), but we don't actually know their status. Here, those tracts still get predictions, but they're **left out of the accuracy calculation**, since there's no true answer to compare against.

In [ ]:
df.isna().sum()

## Step 4: Predict

`model.feature_names_in_` isn't needed here because we saved the feature list in the bundle. Selecting `df[FEATURES]` guarantees the same columns in the same order as training.

The output column names (`pred_prob`, `pred_class`, `risk`) match the logistic regression file, so any later analysis code works on either file.

In [ ]:
X_2025 = df[FEATURES]
probs = model.predict_proba(X_2025)[:, 1]

df["pred_prob"] = probs
df["pred_class"] = (probs >= THRESHOLD).astype(int)

df["risk"] = pd.cut(
    df["pred_prob"],
    bins=[0, 0.33, 0.67, 1],
    labels=["low-risk", "medium-risk", "high-risk"],
    include_lowest=True,
)

print(df["risk"].value_counts())

## Step 5: Evaluate on tracts with known labels

In [ ]:
labeled = df["FD_2"].notna()
y_true = df.loc[labeled, "FD_2"].astype(int)
y_pred = df.loc[labeled, "pred_class"]

print(f"Evaluating on {labeled.sum():,} labeled tracts ({(~labeled).sum()} unlabeled skipped)")
print("Accuracy :", accuracy_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))  # rows = actual, cols = predicted
print()
print(classification_report(y_true, y_pred, digits=3))

## Step 6: Compare with logistic regression

This loads the logistic regression's saved 2025 predictions and scores both models on the **same labeled tracts**, matched by `tract_id`.

**Why recall matters here:** Your logistic regression scored 83% accuracy on 2025, but it caught only 1,410 of 13,355 food-desert tracts (about 11% recall). Only about 16% of 2025 tracts are food deserts, compared with about 34% in 2019, so a model that mostly predicts "not a food desert" still gets high accuracy. Recall, F1, and average precision show whether a model actually finds food deserts.

**Distribution shift:** The 2019 data uses 2010 census tracts and the 2025 data uses 2020 tracts, and food-desert rates changed a lot between the two. Both models were trained on 2019 patterns, so some drop in performance on 2025 is expected for either one.

In [ ]:
lr = pd.read_csv("../logistic_regression_modeling/cleaned_2025_w_predictions.csv",
                 usecols=["tract_id", "pred_prob", "pred_class"])
lr = lr.rename(columns={"pred_prob": "lr_prob", "pred_class": "lr_class"})

compare = (df.loc[labeled, ["tract_id", "FD_2", "pred_prob", "pred_class"]]
             .merge(lr, on="tract_id", how="inner"))
y = compare["FD_2"].astype(int)

def scores(y_true, y_class, y_prob):
    return {
        "accuracy":  accuracy_score(y_true, y_class),
        "precision": precision_score(y_true, y_class, zero_division=0),
        "recall":    recall_score(y_true, y_class),
        "f1":        f1_score(y_true, y_class),
        "avg_precision": average_precision_score(y_true, y_prob),
    }

comparison = pd.DataFrame({
    "Logistic regression": scores(y, compare["lr_class"], compare["lr_prob"]),
    "Random forest":       scores(y, compare["pred_class"], compare["pred_prob"]),
}).round(3)

print(f"Compared on {len(compare):,} tracts")
comparison

## Step 7: Save the predictions

In [ ]:
df.to_csv("cleaned_2025_w_rf_predictions.csv", index=False)
print("Saved cleaned_2025_w_rf_predictions.csv")